# IEEE-CIS Fraud Detection — EDA

Goal: understand the data well enough to build a good graph.

Key questions to answer:
1. What is the fraud rate overall and by card/device/email?
2. Which features have high missingness?
3. What does the transaction amount distribution look like for fraud vs non-fraud?
4. Are there cards/emails that appear in many fraud transactions? (→ graph hotspots)
5. What is the time distribution of fraud?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import sys
sys.path.append('..')
from src.data.load import load_raw, merge_tables, get_feature_groups

pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

## 1. Load data

In [ ]:
DATA_DIR = Path('../data/raw')
txn, identity = load_raw(DATA_DIR)
df = merge_tables(txn, identity)
df.head(3)

## 2. Fraud rate overview

In [ ]:
fraud_counts = df['isFraud'].value_counts()
print(f"Not fraud: {fraud_counts[0]:,}  ({fraud_counts[0]/len(df)*100:.1f}%)")
print(f"Fraud:     {fraud_counts[1]:,}  ({fraud_counts[1]/len(df)*100:.1f}%)")

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(['Not Fraud', 'Fraud'], fraud_counts.values, color=['steelblue', 'tomato'])
ax.set_ylabel('Count')
ax.set_title('Class distribution')
plt.tight_layout()
plt.show()

## 3. Transaction amount: fraud vs non-fraud

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for label, grp in df.groupby('isFraud'):
    axes[0].hist(grp['TransactionAmt'].clip(upper=1000), bins=50,
                 alpha=0.6, label='Fraud' if label else 'Not Fraud', density=True)
axes[0].set_title('Transaction amount (clipped at $1000)')
axes[0].set_xlabel('Amount ($)')
axes[0].legend()

for label, grp in df.groupby('isFraud'):
    axes[1].hist(np.log1p(grp['TransactionAmt']), bins=50,
                 alpha=0.6, label='Fraud' if label else 'Not Fraud', density=True)
axes[1].set_title('Log(1 + Transaction amount)')
axes[1].set_xlabel('log1p(Amount)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Fraud rate by card and email — finding graph hotspots

Nodes with high fraud rates will be important in the graph. Find them here.

In [ ]:
def fraud_rate_by(col, min_count=50):
    """Fraud rate per unique value of col, filtered by minimum occurrence count."""
    grp = df.groupby(col)['isFraud'].agg(['mean', 'count'])
    grp.columns = ['fraud_rate', 'count']
    return grp[grp['count'] >= min_count].sort_values('fraud_rate', ascending=False)

print("=== card4 (card network) ===")
print(fraud_rate_by('card4').head(10))

print("\n=== P_emaildomain (purchaser email) ===")
print(fraud_rate_by('P_emaildomain').head(10))

print("\n=== DeviceType ===")
print(fraud_rate_by('DeviceType').head(10))

## 5. Missingness heatmap

In [ ]:
groups = get_feature_groups(df)
miss_pct = df.isnull().mean() * 100

for group_name, cols in groups.items():
    cols_present = [c for c in cols if c in df.columns]
    if not cols_present:
        continue
    miss = miss_pct[cols_present].sort_values(ascending=False)
    if miss.max() < 1:
        continue

    fig, ax = plt.subplots(figsize=(max(6, len(cols_present) * 0.4), 3))
    ax.bar(range(len(miss)), miss.values, color='steelblue')
    ax.set_xticks(range(len(miss)))
    ax.set_xticklabels(miss.index, rotation=90, fontsize=8)
    ax.set_ylabel('Missing (%)')
    ax.set_title(f'Missingness — {group_name} features')
    plt.tight_layout()
    plt.show()

## 6. Time pattern of fraud

In [ ]:
df['hour'] = (df['TransactionDT'] // 3600) % 24
df['day']  = (df['TransactionDT'] // (3600 * 24)) % 7

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for label, grp in df.groupby('isFraud'):
    hourly = grp['hour'].value_counts().sort_index()
    axes[0].plot(hourly.index, hourly.values / hourly.values.sum(),
                 label='Fraud' if label else 'Not Fraud', alpha=0.8)
axes[0].set_title('Transaction hour distribution')
axes[0].set_xlabel('Hour of day')
axes[0].legend()

for label, grp in df.groupby('isFraud'):
    daily = grp['day'].value_counts().sort_index()
    axes[1].plot(daily.index, daily.values / daily.values.sum(),
                 label='Fraud' if label else 'Not Fraud', alpha=0.8)
axes[1].set_title('Transaction day-of-week distribution')
axes[1].set_xlabel('Day (0=Mon)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Key takeaways for graph construction

Fill this in after running the above cells:

- Fraud rate: ____%
- High-fraud email domains: ____
- High-fraud card types: ____
- Missingness: identity features missing ____% for non-identity transactions
- Time pattern: fraud peaks at hour ____

→ These insights should guide which node types matter most in your graph.